# 03. Pelatihan Model Baseline dan Model Gabungan

Model yang dilatih, dari yang paling sederhana hingga yang menggunakan fitur berita:

| Kelompok | Model | Input |
|---|---|---|
| Baseline naive | Majority, Persistence | tanpa pelatihan |
| Deret waktu | ARIMA | perubahan kurs beberapa hari terakhir |
| Harga saja | LR, HGB | 11 fitur harga |
| Berita saja | LR | fitur NLP |
| Gabungan | LR, HGB | fitur harga dan fitur NLP |

Keterangan:
- LR (Logistic Regression): model linear yang menjumlahkan pengaruh setiap fitur.
- HGB (HistGradientBoosting): kumpulan pohon keputusan dari scikit-learn dengan prinsip yang sama seperti XGBoost.

Agar evaluasi tetap objektif:
1. Pengaturan model (hyperparameter) dipilih hanya berdasarkan hasil pada data validation.
2. Model akhir dilatih ulang menggunakan train dan validation, lalu diuji satu kali pada data test.

Hasil prediksi seluruh model disimpan di `data/predictions.csv`.

In [1]:
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")
DATA = Path("../data") if Path("../data").exists() else Path("data") 

PRICE = ["ret_1", "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4",
         "ret_5", "ret_20", "vol_5", "vol_20", "z_20", "gap_days"]
feat = pd.read_csv(DATA / "features.csv", parse_dates=["date"]).sort_values("date")


def load(lag=0):
    """Gabungkan data split dengan fitur NLP. lag=1 berarti memakai berita hari sebelumnya."""
    f = feat.copy()
    if lag:
        cols = f.columns[1:]
        f[cols] = f[cols].astype(float).shift(lag)
    return {s: pd.read_csv(DATA / f"{s}.csv", parse_dates=["date"]).merge(f, on="date", how="left").fillna(0)
            for s in ["train", "val", "test"]}


D = load()
print({s: len(d) for s, d in D.items()})

{'train': 825, 'val': 176, 'test': 178}


## 1. Fungsi Pendukung
Metrik evaluasi yang digunakan:
- DA (directional accuracy): persentase prediksi arah yang benar.
- BalAcc dan F1-macro: ukuran akurasi yang memberi bobot seimbang pada kelas naik dan turun.
- MCC: bernilai 0 untuk prediksi acak atau konstan, dan mendekati 1 untuk prediksi yang sangat baik.
- AUC: kemampuan model membedakan hari naik dan turun. Nilai 0,5 setara dengan prediksi acak.

In [2]:
def metrics(y, pred, score=None):
    out = {"DA": accuracy_score(y, pred), "BalAcc": balanced_accuracy_score(y, pred),
           "F1_macro": f1_score(y, pred, average="macro"), "MCC": matthews_corrcoef(y, pred)}
    out["AUC"] = roc_auc_score(y, score) if score is not None else np.nan
    return out


GRID = {
    "LR": [{"C": c} for c in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]],
    "HGB": [dict(max_depth=d, learning_rate=l, max_iter=i, min_samples_leaf=m)
            for d, l, i, m in product([2, 3], [0.03, 0.1], [100, 300], [20, 50])],
}


def make_model(kind, p):
    if kind == "LR":
        return make_pipeline(StandardScaler(), LogisticRegression(**p, class_weight="balanced", max_iter=5000))
    return HistGradientBoostingClassifier(**p, l2_regularization=1.0, class_weight="balanced", random_state=42)


def tune(kind, d, cols):
    """Coba semua hyperparameter: latih di train, nilai di validation, ambil F1-macro terbaik."""
    best = None
    for p in GRID[kind]:
        prob = make_model(kind, p).fit(d["train"][cols], d["train"]["target"]).predict_proba(d["val"][cols])[:, 1]
        sc = metrics(d["val"]["target"], (prob >= 0.5).astype(int), prob)
        if best is None or sc["F1_macro"] > best[1]["F1_macro"]:
            best = (p, sc, prob)
    return best


def run(name, kind, d, cols):
    """Tuning di validation, lalu latih ulang di train+val dan prediksi test."""
    p, sc_val, prob_val = tune(kind, d, cols)
    trval = pd.concat([d["train"], d["val"]])
    prob_test = make_model(kind, p).fit(trval[cols], trval["target"]).predict_proba(d["test"][cols])[:, 1]
    save(name, (prob_val >= 0.5).astype(int), prob_val, (prob_test >= 0.5).astype(int), prob_test)
    print(f"{name:25s} parameter={p}")


RES = {"val": {}, "test": {}}  # menyimpan prediksi seluruh model


def save(name, pred_val, score_val, pred_test, score_test):
    RES["val"][name] = (pred_val, score_val)
    RES["test"][name] = (pred_test, score_test)

## 2. Baseline Naive
- Majority: selalu memprediksi naik karena kelas tersebut paling sering muncul pada data train.
- Persistence: memprediksi arah hari berikutnya sama dengan arah hari ini.

Model yang lebih kompleks baru dianggap bermanfaat apabila mampu mengungguli kedua baseline ini.

In [3]:
for s in ["val", "test"]:
    RES[s]["Naive: Majority"] = (np.ones(len(D[s]), dtype=int), None)
    RES[s]["Naive: Persistence"] = ((D[s]["ret_1"] > 0).astype(int).values, None)

## 3. ARIMA
ARIMA memprediksi perubahan kurs hari berikutnya berdasarkan perubahan kurs beberapa hari terakhir.
Jumlah hari yang digunakan (p, antara 0 sampai 5) dipilih berdasarkan nilai AIC terkecil pada data train.
Jika perubahan yang diprediksi bernilai positif, prediksi arahnya adalah NAIK.

Tabel di bawah juga membandingkan galat prediksi kurs (dalam Rupiah) dengan random walk,
yaitu asumsi bahwa kurs besok sama dengan kurs hari ini.

In [4]:
LAGS = ["ret_1", "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4"]


def fit_ar(d, p):
    X = np.column_stack([np.ones(len(d))] + [d[c] for c in LAGS[:p]])
    beta, *_ = np.linalg.lstsq(X, d["target_ret"], rcond=None)
    resid = d["target_ret"] - X @ beta
    return beta, len(d) * np.log(resid.var()) + 2 * (p + 1)


def predict_ar(d, beta, p):
    X = np.column_stack([np.ones(len(d))] + [d[c] for c in LAGS[:p]])
    return X @ beta


aic = {p: fit_ar(D["train"], p)[1] for p in range(6)}
P = min(aic, key=aic.get)
print("AIC:", {k: round(v, 1) for k, v in aic.items()}, "-> orde terpilih p =", P)

ARIMA_NAME = f"ARIMA({P},1,0)"
r_val = predict_ar(D["val"], fit_ar(D["train"], P)[0], P)
r_test = predict_ar(D["test"], fit_ar(pd.concat([D["train"], D["val"]]), P)[0], P)
save(ARIMA_NAME, (r_val > 0).astype(int), r_val, (r_test > 0).astype(int), r_test)

# Bandingkan galat level kurs (Rupiah) dengan random walk (kurs besok = kurs hari ini)
rows = []
for s, r_hat in [("val", r_val), ("test", r_test)]:
    actual = D[s]["usd_idr"] * np.exp(D[s]["target_ret"])
    for name, pred in [("Random walk", D[s]["usd_idr"]), (ARIMA_NAME, D[s]["usd_idr"] * np.exp(r_hat))]:
        err = actual - pred
        rows.append({"split": s, "model": name, "RMSE (Rp)": np.sqrt((err ** 2).mean()), "MAE (Rp)": err.abs().mean()})
pd.DataFrame(rows).round(2)

AIC: {0: np.float64(-9357.6), 1: np.float64(-9365.7), 2: np.float64(-9377.7), 3: np.float64(-9377.2), 4: np.float64(-9376.6), 5: np.float64(-9374.6)} -> orde terpilih p = 2


,split,model,RMSE (Rp),MAE (Rp)
0,val,Random walk,55.93,40.78
1,val,"ARIMA(2,1,0)",55.76,41.34
2,test,Random walk,51.19,38.60
3,test,"ARIMA(2,1,0)",50.99,38.24


## 4. Eksperimen A: Perbandingan Sumber Teks
Model LR dengan fitur harga dan fitur NLP diuji untuk setiap sumber teks (judul, paragraf awal, dan isi lengkap)
serta setiap jenis fitur (leksikon, TF-IDF, atau keduanya). Seluruh kombinasi dinilai pada data validation,
dan kombinasi dengan F1-macro tertinggi digunakan pada tahap berikutnya.

In [5]:
def nlp_cols(src, kind):
    prefixes = {"lex": [f"lex_{src}_"], "tfidf": [f"tfidf_{src}_"], "all": [f"lex_{src}_", f"tfidf_{src}_"]}[kind]
    return [c for c in feat.columns if any(c.startswith(p) for p in prefixes)]


rows = []
for src, kind in product(["title", "lead", "full"], ["lex", "tfidf", "all"]):
    _, sc, _ = tune("LR", D, PRICE + nlp_cols(src, kind))
    rows.append({"sumber": src, "fitur_nlp": kind, **sc})
expA = pd.DataFrame(rows)
best = expA.sort_values(["F1_macro", "AUC"], ascending=False).iloc[0]
SRC, KIND = best["sumber"], best["fitur_nlp"]
print(f"Terpilih: sumber = {SRC}, fitur = {KIND}")
expA.round(3)

Terpilih: sumber = full, fitur = lex


,sumber,fitur_nlp,DA,BalAcc,F1_macro,MCC,AUC
0,title,lex,0.489,0.492,0.485,-0.016,0.503
1,title,tfidf,0.472,0.479,0.450,-0.047,0.504
2,title,all,0.500,0.508,0.473,0.019,0.512
3,lead,lex,0.500,0.503,0.498,0.005,0.484
4,lead,tfidf,0.500,0.503,0.498,0.006,0.472
5,lead,all,0.494,0.497,0.493,-0.007,0.464
6,full,lex,0.517,0.516,0.516,0.032,0.474
7,full,tfidf,0.438,0.441,0.432,-0.120,0.500
8,full,all,0.511,0.512,0.511,0.023,0.518


## 5. Model Harga Saja, NLP Saja, dan Gabungan
Seluruh model menggunakan sumber teks yang terpilih pada Eksperimen A.
Output di bawah menampilkan hyperparameter terbaik untuk setiap model.

In [6]:
NLP = nlp_cols(SRC, KIND)
run("LR [Harga]", "LR", D, PRICE)
run("HGB [Harga]", "HGB", D, PRICE)
run("LR [NLP]", "LR", D, NLP)
run("LR [Harga+NLP]", "LR", D, PRICE + NLP)
run("HGB [Harga+NLP]", "HGB", D, PRICE + NLP)

LR [Harga]                parameter={'C': 0.3}
HGB [Harga]               parameter={'max_depth': 3, 'learning_rate': 0.03, 'max_iter': 300, 'min_samples_leaf': 50}
LR [NLP]                  parameter={'C': 0.01}
LR [Harga+NLP]            parameter={'C': 0.03}
HGB [Harga+NLP]           parameter={'max_depth': 3, 'learning_rate': 0.1, 'max_iter': 300, 'min_samples_leaf': 50}


## 6. Eksperimen B: Model Gabungan dengan Fitur NLP Terpilih (top-k)
Model gabungan sebelumnya menggunakan banyak fitur NLP, sedangkan data train hanya berjumlah 825 hari.
Kondisi ini berisiko menyebabkan overfitting, yaitu model terlalu menyesuaikan diri dengan data latih
sehingga kinerjanya menurun pada data baru.

Pada eksperimen ini hanya digunakan k fitur leksikon (k = 3, 5, atau 10) yang korelasinya dengan
perubahan kurs hari berikutnya paling kuat pada data train.

In [7]:
lex_all = [c for c in feat.columns if c.startswith("lex_")]
rho = D["train"][lex_all].corrwith(D["train"]["target_ret"], method="spearman").abs().sort_values(ascending=False)

best_k = max([3, 5, 10], key=lambda k: tune("LR", D, PRICE + list(rho.index[:k]))[1]["F1_macro"])
TOPK = list(rho.index[:best_k])
run("LR [Harga+NLP top-k]", "LR", D, PRICE + TOPK)
print(f"k = {best_k}, fitur terpilih: {TOPK}")

LR [Harga+NLP top-k]      parameter={'C': 0.003}
k = 3, fitur terpilih: ['lex_full_vader_neu', 'lex_lead_vader_pos', 'lex_title_lm_uncertainty_dev']


## 7. Uji Sensitivitas: Fitur Berita Digeser Satu Hari (lag-1)
Sebagian besar berita hanya memiliki tanggal terbit tanpa keterangan jam. Akibatnya, ada kemungkinan sebagian berita
sebenarnya terbit setelah kurs terbentuk. Untuk mengujinya, model menggunakan fitur berita dari hari sebelumnya.
Jika kinerja tidak menurun, hasil model tidak bergantung pada berita yang waktu terbitnya tidak pasti.

In [8]:
D_lag = load(lag=1)
run("LR [Harga+NLP, lag-1]", "LR", D_lag, PRICE + NLP)
run("HGB [Harga+NLP, lag-1]", "HGB", D_lag, PRICE + NLP)

LR [Harga+NLP, lag-1]     parameter={'C': 1.0}
HGB [Harga+NLP, lag-1]    parameter={'max_depth': 3, 'learning_rate': 0.1, 'max_iter': 100, 'min_samples_leaf': 50}


## 8. Ringkasan Hasil
Tabel pertama menampilkan hasil pada data validation (model dilatih dengan data train).

In [9]:
def table(s):
    y = D[s]["target"]
    return pd.DataFrame({m: metrics(y, p, sc) for m, (p, sc) in RES[s].items()}).T.round(3)


# Validation (model dilatih di train)
table("val")

,DA,BalAcc,F1_macro,MCC,AUC
Naive: Majority,0.517,0.500,0.341,0.000,NaN
Naive: Persistence,0.517,0.516,0.516,0.033,NaN
"ARIMA(2,1,0)",0.483,0.479,0.474,-0.043,0.496
LR [Harga],0.506,0.508,0.504,0.016,0.501
HGB [Harga],0.523,0.521,0.520,0.042,0.503
LR [NLP],0.477,0.473,0.466,-0.056,0.435
LR [Harga+NLP],0.517,0.516,0.516,0.032,0.474
HGB [Harga+NLP],0.517,0.516,0.515,0.031,0.471
LR [Harga+NLP top-k],0.545,0.546,0.545,0.092,0.528
"LR [Harga+NLP, lag-1]",0.494,0.496,0.494,-0.008,0.499


Tabel kedua menampilkan hasil pada data test (model dilatih dengan train dan validation, lalu diuji satu kali).
Sebagai pembanding, prediksi "selalu NAIK" sudah memperoleh DA 0,579 pada data test.

In [10]:
table("test")

,DA,BalAcc,F1_macro,MCC,AUC
Naive: Majority,0.579,0.500,0.367,0.000,NaN
Naive: Persistence,0.545,0.532,0.533,0.065,NaN
"ARIMA(2,1,0)",0.562,0.540,0.538,0.082,0.516
LR [Harga],0.517,0.515,0.513,0.031,0.527
HGB [Harga],0.500,0.503,0.498,0.005,0.538
LR [NLP],0.517,0.506,0.506,0.013,0.506
LR [Harga+NLP],0.517,0.517,0.514,0.034,0.539
HGB [Harga+NLP],0.455,0.457,0.453,-0.086,0.491
LR [Harga+NLP top-k],0.539,0.542,0.537,0.083,0.575
"LR [Harga+NLP, lag-1]",0.528,0.514,0.514,0.029,0.505


## 9. Menyimpan Hasil Prediksi
Hasil prediksi seluruh model disimpan untuk dievaluasi pada notebook 04.

In [11]:
out = []
for s in ["val", "test"]:
    df = D[s][["date", "usd_idr", "target"]].assign(split=s)
    for m, (p, sc) in RES[s].items():
        df[f"pred::{m}"] = p
        if sc is not None:
            df[f"score::{m}"] = sc
    out.append(df)
pd.concat(out).to_csv(DATA / "predictions.csv", index=False, date_format="%Y-%m-%d")
print("Tersimpan data/predictions.csv")

Tersimpan data/predictions.csv
